In [ ]:
Задание:
- генерация аудио (AudioLDM2 from diffusers + аналоги)
- типы задач с моделями и реализациями (только предобученные)
- voice fake (как работает, архитектуры, реализация, попробовать готовые модели)

Генерация аудио (text-to-audio)
- audioldm2

In [8]:
import scipy
import torch
from diffusers import AudioLDM2Pipeline
from transformers import GPT2LMHeadModel

In [7]:
out_path = 'out/'

In [ ]:
repo_id = "cvssp/audioldm2"

language_model = GPT2LMHeadModel.from_pretrained(
    repo_id, 
    subfolder="language_model", 
    torch_dtype=torch.float16
)

pipe = AudioLDM2Pipeline.from_pretrained(
    repo_id, 
    language_model=language_model,
    torch_dtype=torch.float16
)
pipe = pipe.to("cuda")

prompt = "A thunderclap in rainy weather."
negative_prompt = "Low quality."

generator = torch.Generator("cuda").manual_seed(0)

original_get_text_features = pipe.text_encoder.get_text_features

def patched_get_text_features(input_ids, attention_mask=None, **kwargs):
    text_outputs = pipe.text_encoder.text_model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        **kwargs
    )
    pooled_output = text_outputs.pooler_output if hasattr(text_outputs, "pooler_output") else text_outputs[1]

    text_embeds = pipe.text_encoder.text_projection(pooled_output)
    return text_embeds

pipe.text_encoder.get_text_features = patched_get_text_features

audio = pipe(
    prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=200,
    audio_length_in_s=10.0,
    num_waveforms_per_prompt=3,
    generator=generator,
).audios

scipy.io.wavfile.write(out_path + "lightning.wav", rate=16000, data=audio[0])

100%|██████████| 200/200 [00:20<00:00,  9.86it/s]


In [ ]:
# musicgen-small

In [9]:
from transformers import AutoProcessor, MusicgenForConditionalGeneration

In [11]:
processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small")
model = model.to("cuda")

prompts = ["A dynamic electronic dance track with synth bass and driving drums"]

out = out_path + "musicgen_dance.wav"

def get_audio(promt, out_path):
    inputs = processor(text=promt, padding=True, return_tensors="pt").to("cuda")

    audio_values = model.generate(**inputs, max_new_tokens=512)

    sampling_rate = model.config.audio_encoder.sampling_rate

    audio_data = audio_values[0, 0].cpu().numpy()

    scipy.io.wavfile.write(out_path, rate=sampling_rate, data=audio_data)

get_audio(prompts, out)

Loading weights: 100%|██████████| 611/611 [00:00<00:00, 814.74it/s, Materializing param=text_encoder.shared.weight]                                                       
MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-small
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
prompts = ["A thunderclap in rainy weather."]
out = out_path + "lighting_2.wav"

get_audio(prompts, out)

### Типы задач с аудио

**1. Синтез речи (Text-to-Speech / TTS):**
<br>Самая старая и развитая область. Задача — преобразовать написанный текст в человеческую речь
- Обычный TTS: Генерация голоса диктора по тексту (например, для аудиокниг, навигаторов, голосовых ассистентов);
- Voice Cloning (Клонирование голоса): Модель получает короткий сэмпл голоса конкретного человека (обычно 5–10 секунд) и генерирует новый текст так, будто его читает этот человек;
- Expressive TTS: Генерация речи с заданными эмоциями, акцентами, интонациями или пением (например, Bark от Suno).

**2. Генерация музыки:**
<br>Создание музыкальных композиций. Это очень сложная задача, так как музыка имеет сложную структуру (ритм, гармония, мелодия)
- Text-to-Music: Создание музыки по текстовому описанию (например, "джазовая композиция 80-х с саксофоном"). Эту задачу решают модели вроде MusicGen;
- Audio-to-Audio (Style Transfer): Перенос музыкального стиля. Например, вы напеваете мелодию голосом на диктофон, а модель превращает ее в соло на электрогитаре или полноценный оркестровый трек;
- Continuation / Outpainting: Модели дают начало трека (например, первые 10 секунд), а она логично продолжает композицию дальше.

**3. Генерация звуковых эффектов:**
<br>Создание фоновых шумов, звуков природы или специфических эффектов для кино и видеоигр
- Text-to-Audio (SFX): Создание звукововых эффектов из текстового описания. Отлично решается моделями AudioGen и Stable Audio Open;
- Video-to-Audio (Foley Generation): Модель анализирует беззвучный видеоряд (например, человек идет по снегу) и автоматически генерирует синхронизированные звуки шагов и хруста.

**4. Редактирование и восстановление аудио:**
<br>Модели, которые не создают звук с нуля, а генерируют недостающие или измененные части поверх существующего аудио
- Audio Inpainting: Восстановление поврежденного фрагмента записи (например, если в середине трека был сбой или резкий шум, модель генерирует этот кусочек заново, чтобы он слился с оригиналом);
- Speech Enhancement / Denoising: Генеративное удаление шумов. В отличие от старых фильтров, ИИ не просто вырезает частоты, а "перерисовывает" голос так, чтобы он звучал чисто, как из студии;
- Source Separation (Разделение источников): Извлечение отдельных инструментов (например, только вокал или только барабаны) из готового трека.

text-to-speech (генерация речи)

In [2]:
import soundfile as sf
import torch
from transformers import pipeline
from datasets import load_dataset

In [ ]:
synthesiser = pipeline("text-to-speech", model="microsoft/speecht5_tts")

torch.manual_seed(0)
speaker_embedding = torch.randn(1, 512)

text = '''
    On a summer morning, three piglets frolicked at the edge of the forest. They jumped in puddles and sang so carefree, as if summer would never end.
    Only now it was getting colder in the evenings, the brothers were freezing, and the youngest offered to build a strong, warm house.
    The idea that it was time to stop playing upset the other two piglets so much that it was decided to postpone the construction until later.
    It was only when the cold became unbearable that our story began.
    '''

speech = synthesiser(text, forward_params={"speaker_embeddings": speaker_embedding})

sf.write(out_path + "speech_output.wav", speech["audio"], samplerate=speech["sampling_rate"])

Loading weights: 100%|██████████| 396/396 [00:00<00:00, 999.24it/s, Materializing param=speecht5.encoder.wrapped_encoder.layers.11.layer_norm.weight]                     
SpeechT5ForTextToSpeech LOAD REPORT from: microsoft/speecht5_tts
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
speecht5.encoder.prenet.encode_positions.pe | UNEXPECTED |  | 
speecht5.decoder.prenet.encode_positions.pe | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 158/158 [00:00<00:00, 1048.67it/s, Materializing param=upsampler.3.weight]          


### Voice deepfake

**Пайплайн:**
- Speaker Encoder (Кодировщик голоса): Принимает короткий аудиофайл с образцом речи и преобразует его в эмбеддинг — векторное представление фиксированной размерности. Этот математический «слепок» содержит информацию о тембре и особенностях голоса, отделенную от произносимого текста;
- Synthesizer (Синтезатор): Принимает на вход текст и полученный вектор голоса, после чего генерирует промежуточное представление звука — мел-спектрограмму (визуальное отображение частот во времени). В этой роли часто выступают модели Tacotron 2 или FastSpeech;
- Vocoder (Вокодер): Трансформирует сгенерированную спектрограмму в реальную звуковую волну. Современные генеративно-состязательные вокодеры, такие как HiFi-GAN или WaveNet, устраняют роботизированные искажения и делают звук максимально естественным.

**Современные архитектуры:**
- RVC (Retrieval-based Voice Conversion): Популярная архитектура для преобразования речи в речь (Speech-to-Speech). Она использует предобученную модель HuBERT для извлечения акустических признаков из исходного голоса, алгоритмы извлечения высоты тона (pitch) и генератор на базе HiFi-GAN. Главная особенность RVC — механизм поиска (retrieval) по индексу обучающих данных, что позволяет сохранять вокальные нюансы целевого голоса без генерации звука с нуля;<br>RVC не превращает текст в звук, она берет уже готовую аудиозапись и заменяет в ней голос одного человека на голос другого
- XTTS и VALL-E: Модели нового поколения, использующие архитектуру Transformer и принципы языкового моделирования (Language Modeling) для генерации речи из текста. Например, XTTS применяет VQ-VAE для преобразования звука в дискретные токены, а затем GPT-подобная модель предсказывает аудиотокены на основе текста и характеристик голоса. VALL-E позволяет клонировать голос методом Zero-Shot всего по трехсекундному образцу;
- VITS: End-to-end архитектура, которая объединяет синтез спектрограмм и вокодер в одну сквозную нейросеть. Использование вариационного вывода делает генерацию речи более плавной и быстрой, что идеально подходит для работы с минимальной задержкой.

**Реализации:**
1. Zero-Shot клонирование: Система не требует долгого обучения. В интерфейс программы загружается короткий пример голоса (3–10 секунд), энкодер мгновенно извлекает эмбеддинг, и нейросеть сразу синтезирует нужный текст с похожим тембром. Этот метод быстрый, но может не передавать 100% интонационных привычек человека;
2. Дообучение (Fine-Tuning): Для создания идеальной копии собирается очищенный от шумов датасет записей целевого голоса (от 5 до 30 минут). На этих данных дообучается базовая нейросеть (например, в интерфейсах RVC WebUI или SO-VITS-SVC). Процесс требует вычислительных мощностей видеокарты и занимает некоторое время, однако итоговая модель способна безупречно имитировать даже пение с сохранением дыхания и акцента.

In [ ]:
# https://github.com/jamiepine/voicebox